# boto3 S3 Upload Test

Tests direct S3 uploads to MinIO (`s3.nevaobjects.id`) to isolate the `XAmzContentSHA256Mismatch` error.

**Root cause**: boto3 ≥1.34 sends CRC32 checksum headers by default (`request_checksum_calculation=when_supported`).  
MinIO rejects these with `XAmzContentSHA256Mismatch`.  
Note: `payload_signing_enabled=True` alone does NOT fix this — the checksum config is what matters.

**Fix**: `request_checksum_calculation='when_required'` + `response_checksum_validation='when_required'` in `botocore.config.Config`,  
or equivalently set env vars `AWS_REQUEST_CHECKSUM_CALCULATION=when_required` / `AWS_RESPONSE_CHECKSUM_VALIDATION=when_required`.

In [8]:
import boto3
import botocore
from botocore.config import Config
import tempfile
import os

# ── Credentials ───────────────────────────────────────────────────────────────
AWS_ACCESS_KEY_ID     = "BQFNKRBSTNFJFA53UY08"
AWS_SECRET_ACCESS_KEY = "XXMRRKJxqJQkk6L3b4s26YStj71Fvo4toipBuOp2"
ENDPOINT_URL          = "https://s3.nevaobjects.id"
BUCKET                = "research-geoai"
TEST_PREFIX           = "mlflow/artifacts/test-boto3-direct"

# ── boto3 config that works with MinIO ────────────────────────────────────────
MINIO_CONFIG = Config(
    request_checksum_calculation="when_required",
    response_checksum_validation="when_required",
)

print(f"Endpoint : {ENDPOINT_URL}")
print(f"Bucket   : {BUCKET}")
print(f"boto3    : {boto3.__version__}")
print(f"botocore : {botocore.__version__}")

Endpoint : https://s3.nevaobjects.id
Bucket   : research-geoai
boto3    : 1.42.60
botocore : 1.42.60


## 1. List bucket (connection check)

In [9]:
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

try:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="mlflow/", MaxKeys=5)
    keys = [o["Key"] for o in resp.get("Contents", [])]
    print(f"[OK] Connected — {len(keys)} objects (first 5 under mlflow/):")
    for k in keys:
        print(f"  {k}")
except Exception as e:
    print(f"[ERROR] {e}")

[OK] Connected — 3 objects (first 5 under mlflow/):
  mlflow/artifacts/test-boto3-direct/fixed.txt
  mlflow/test-direct/checksum_when_req.txt
  mlflow/test-direct/path_style.txt


## 2. Upload WITHOUT fix (expected to FAIL — boto3 ≥1.34 checksum issue)

In [10]:
s3_default = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    # No config — boto3 ≥1.34 adds CRC32 checksum headers by default
)

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write("boto3 test — no fix\n")
    tmp_path = f.name

key = f"{TEST_PREFIX}/no_fix.txt"
print(f"Uploading → s3://{BUCKET}/{key}")
try:
    s3_default.upload_file(tmp_path, BUCKET, key)
    print("[OK] Succeeded (unexpected — MinIO accepted CRC32 checksums)")
except Exception as e:
    print(f"[FAIL as expected] {type(e).__name__}: {e}")
finally:
    os.unlink(tmp_path)

Uploading → s3://research-geoai/mlflow/artifacts/test-boto3-direct/no_fix.txt
[FAIL as expected] S3UploadFailedError: Failed to upload /var/folders/hb/b1p7kpgj4s759vkptsfqzxzw0000gn/T/tmpgg_swlgl.txt to research-geoai/mlflow/artifacts/test-boto3-direct/no_fix.txt: An error occurred (XAmzContentSHA256Mismatch) when calling the PutObject operation: None


## 3. Upload WITH fix — `request_checksum_calculation=when_required` (expected to SUCCEED)

In [11]:
s3_fixed = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    config=MINIO_CONFIG,
)

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write("boto3 test — checksum fix applied\n")
    tmp_path = f.name

key = f"{TEST_PREFIX}/fixed.txt"
print(f"Uploading → s3://{BUCKET}/{key}")
try:
    s3_fixed.upload_file(tmp_path, BUCKET, key)
    print("[OK] Upload succeeded")
except Exception as e:
    print(f"[FAIL] {type(e).__name__}: {e}")
finally:
    os.unlink(tmp_path)

Uploading → s3://research-geoai/mlflow/artifacts/test-boto3-direct/fixed.txt
[OK] Upload succeeded


## 4. Upload binary (PNG) with fix

In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot([1, 2, 3], [0.4, 0.6, 0.75], marker="o", label="mIoU")
ax.set_title("Test plot")
ax.legend()
fig.tight_layout()

with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
    fig.savefig(f.name, dpi=80)
    plt.close(fig)
    tmp_png = f.name

key = f"{TEST_PREFIX}/test_plot.png"
print(f"Uploading binary → s3://{BUCKET}/{key}")
try:
    s3_fixed.upload_file(tmp_png, BUCKET, key)
    print("[OK] Binary upload succeeded")
except Exception as e:
    print(f"[FAIL] {type(e).__name__}: {e}")
finally:
    os.unlink(tmp_png)

Uploading binary → s3://research-geoai/mlflow/artifacts/test-boto3-direct/test_plot.png
[OK] Binary upload succeeded


## 5. Verify uploaded objects exist

In [13]:
resp = s3_fixed.list_objects_v2(Bucket=BUCKET, Prefix=TEST_PREFIX)
objects = resp.get("Contents", [])
print(f"Objects under s3://{BUCKET}/{TEST_PREFIX}:")
for o in objects:
    print(f"  {o['Key']}  ({o['Size']} bytes)")
if not objects:
    print("  (none found)")

Objects under s3://research-geoai/mlflow/artifacts/test-boto3-direct:
  mlflow/artifacts/test-boto3-direct/fixed.txt  (36 bytes)
  mlflow/artifacts/test-boto3-direct/test_plot.png  (9986 bytes)


## 6. Cleanup

In [14]:
DRY_RUN = True

resp = s3_fixed.list_objects_v2(Bucket=BUCKET, Prefix=TEST_PREFIX)
objects = resp.get("Contents", [])

for o in objects:
    if DRY_RUN:
        print(f"[DRY RUN] would delete: {o['Key']}")
    else:
        s3_fixed.delete_object(Bucket=BUCKET, Key=o["Key"])
        print(f"[DELETED] {o['Key']}")

if DRY_RUN:
    print("\nSet DRY_RUN=False to actually delete.")

[DRY RUN] would delete: mlflow/artifacts/test-boto3-direct/fixed.txt
[DRY RUN] would delete: mlflow/artifacts/test-boto3-direct/test_plot.png

Set DRY_RUN=False to actually delete.
